# CS 898BA Project: Diabetic Retinopathy Grading### Colab runner notebook**Author:** Abdul Aleem MohammedThis notebook trains the DR grading model on Google Colab. It clones the project repo,downloads the APTOS 2019 dataset, and runs both the baseline and the full imageprocessing pipeline so the two can be compared.**Before you start:** set the runtime to a GPU.`Runtime` -> `Change runtime type` -> `Hardware accelerator: GPU` -> Save.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Clone the project repository

In [ ]:
!git clone https://github.com/thealeemquadri/AbdulAleemMohammed-CS898BA-Project.git%cd AbdulAleemMohammed-CS898BA-Project!pip install -q -r requirements.txt

## 3. Kaggle credentialsGet your token from kaggle.com -> avatar -> Settings -> API -> **Create New Token**.That downloads `kaggle.json`. Run the cell below and upload it.You must also open the[APTOS 2019 competition page](https://www.kaggle.com/competitions/aptos2019-blindness-detection)and click **Join Competition** to accept the rules, or the download will fail with a 403.

In [ ]:
from google.colab import filesprint("Upload your kaggle.json:")files.upload()!mkdir -p ~/.kaggle!cp kaggle.json ~/.kaggle/!chmod 600 ~/.kaggle/kaggle.json!pip install -q kaggleprint("Kaggle credentials installed.")

## 4. Download the APTOS 2019 dataset

In [ ]:
!mkdir -p data!kaggle competitions download -c aptos2019-blindness-detection -p data!unzip -q -o data/aptos2019-blindness-detection.zip -d data!rm -f data/aptos2019-blindness-detection.zipimport pandas as pddf = pd.read_csv("data/train.csv")print("Total images:", len(df))print("\nClass distribution:")print(df["diagnosis"].value_counts().sort_index())

## 5. Sanity check: view the preprocessing pipeline on one real imageThis confirms the image processing works on real data before we spend GPU time.

In [ ]:
import cv2, matplotlib.pyplot as pltfrom src.preprocessing.fundus import preprocess, preprocess_baselinerow = df[df.diagnosis == 3].iloc[0]img = cv2.cvtColor(cv2.imread(f"data/train_images/{row.id_code}.png"), cv2.COLOR_BGR2RGB)base = preprocess_baseline(img)full = preprocess(img)fig, ax = plt.subplots(1, 3, figsize=(15, 5))ax[0].imshow(img);  ax[0].set_title("Raw fundus (grade 3)")ax[1].imshow(base); ax[1].set_title("Baseline: resize only")ax[2].imshow(full); ax[2].set_title("Full processing pipeline")for a in ax: a.axis("off")plt.tight_layout(); plt.show()

## 6. Train the BASELINEResize only, no domain image processing. This is our control.Roughly 20 to 35 minutes on a T4.

In [ ]:
!python -m src.train --config baseline_resize_only --tag baseline --epochs 12 --batch_size 16

## 7. Train the FULL PIPELINESame model, same schedule, same seed. The only thing that changes is the imageprocessing. Any difference in QWK is therefore attributable to the processing.

In [ ]:
!python -m src.train --config full_pipeline --tag full --epochs 12 --batch_size 16

## 8. Compare the two runsThis is the headline result for the midterm: does the domain image processingactually beat a plain resize?

In [ ]:
import jsonbase = json.load(open("outputs/baseline_results.json"))full = json.load(open("outputs/full_results.json"))print(f"{'Metric':<22}{'Baseline':>12}{'Full pipeline':>16}{'Delta':>10}")print("-" * 60)for k, label in [("final_qwk", "QWK (primary)"), ("final_accuracy", "Accuracy")]:    b, f = base[k], full[k]    print(f"{label:<22}{b:>12.4f}{f:>16.4f}{f-b:>+10.4f}")print("\nPer-class sensitivity (recall):")names = ["No DR", "Mild", "Moderate", "Severe", "Proliferative"]print(f"{'Grade':<18}{'Baseline':>12}{'Full':>12}{'Delta':>10}")for i, n in enumerate(names):    b, f = base["sensitivity"][i], full["sensitivity"][i]    print(f"{n:<18}{b:>12.3f}{f:>12.3f}{f-b:>+10.3f}")

## 9. Generate the presentation figures

In [ ]:
!python -m src.figures --results outputs/full_results.json --severity 3from IPython.display import Image, displayfor f in ["figures/pipeline_demo.png", "figures/class_dist.png",          "figures/training_curves.png", "figures/confusion_matrix.png"]:    display(Image(f))

## 10. Download the resultsGrab the figures and result JSONs to build the presentation.

In [ ]:
!zip -r results_bundle.zip figures outputs/*.jsonfrom google.colab import filesfiles.download("results_bundle.zip")